For preprocessing, we will split the original dataset into training (60%), calibration (20%), and testing (20%) sets. We will then create separate dataframes for the global population and each demographic group (income, sex, education, and age) to train and evaluate our global and demographic-specific models.

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

# Download the dataset
cdc_diabetes = fetch_ucirepo(id=891)

# Separate the features and the target
X = cdc_diabetes.data.features
y = cdc_diabetes.data.targets["Diabetes_binary"]

# Combine into one dataframe
df = pd.concat([X, y], axis=1)

# Split into training (60%) and temporary (40%) set to split later
global_train_df, temp_df = train_test_split(
    df,
    test_size=0.40,
    random_state=42,
    stratify=df["Diabetes_binary"]                                      # Stratify so we have a proportional number of positive and negative classifications in each split
)

# Split temporary data into calibration (20%) and testing (20%)
global_cal_df, global_test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["Diabetes_binary"]
)

# Separate features and target for each split
X_train = global_train_df.drop(columns=["Diabetes_binary"])
y_train = global_train_df["Diabetes_binary"]

X_cal = global_cal_df.drop(columns=["Diabetes_binary"])
y_cal = global_cal_df["Diabetes_binary"]

X_test = global_test_df.drop(columns=["Diabetes_binary"])
y_test = global_test_df["Diabetes_binary"]


# Now we create demographic-specific datasets
# We do this by taking the global split dataframes and filtering for only the entries we want


# Sex groups
# In this case, we grab when "Sex" == 0 (female) and when "Sex" == 1 (male)

# Female
female_train = global_train_df[global_train_df["Sex"] == 0]
female_cal = global_cal_df[global_cal_df["Sex"] == 0]
female_test = global_test_df[global_test_df["Sex"] == 0]

# Male
male_train = global_train_df[global_train_df["Sex"] == 1]
male_cal = global_cal_df[global_cal_df["Sex"] == 1]
male_test = global_test_df[global_test_df["Sex"] == 1]


# Age groups
# In this case we create splits for each of the three subgroups
# 1-4 = Younger Adult, 5-8 = Middle Age, 9-13 = Older

younger_train = global_train_df[global_train_df["Age"].between(1, 4)]
younger_cal = global_cal_df[global_cal_df["Age"].between(1, 4)]
younger_test = global_test_df[global_test_df["Age"].between(1, 4)]

middle_train = global_train_df[global_train_df["Age"].between(5, 8)]
middle_cal = global_cal_df[global_cal_df["Age"].between(5, 8)]
middle_test = global_test_df[global_test_df["Age"].between(5, 8)]

older_train = global_train_df[global_train_df["Age"].between(9, 13)]
older_cal = global_cal_df[global_cal_df["Age"].between(9, 13)]
older_test = global_test_df[global_test_df["Age"].between(9, 13)]


# Income groups
# In this case we create splits for each of the three subgroups
# 1-3 = Lower Income, 4-6 = Middle Income, 6-8 = Higher Income

lower_income_train = global_train_df[global_train_df["Income"].between(1, 3)]
lower_income_cal = global_cal_df[global_cal_df["Income"].between(1, 3)]
lower_income_test = global_test_df[global_test_df["Income"].between(1, 3)]

middle_income_train = global_train_df[global_train_df["Income"].between(4, 6)]
middle_income_cal = global_cal_df[global_cal_df["Income"].between(4, 6)]
middle_income_test = global_test_df[global_test_df["Income"].between(4, 6)]

higher_income_train = global_train_df[global_train_df["Income"].between(7, 8)]
higher_income_cal = global_cal_df[global_cal_df["Income"].between(7, 8)]
higher_income_test = global_test_df[global_test_df["Income"].between(7, 8)]

# Education groups
# In this case we create splits for each of the three subgroups
# 1-2 = Below High School, 3-4 = High School, 5-6 = Some College or Higher

below_hs_train = global_train_df[global_train_df["Education"].between(1, 2)]
below_hs_cal = global_cal_df[global_cal_df["Education"].between(1, 2)]
below_hs_test = global_test_df[global_test_df["Education"].between(1, 2)]

hs_train = global_train_df[global_train_df["Education"].between(3, 4)]
hs_cal = global_cal_df[global_cal_df["Education"].between(3, 4)]
hs_test = global_test_df[global_test_df["Education"].between(3, 4)]

college_train = global_train_df[global_train_df["Education"].between(5, 6)]
college_cal = global_cal_df[global_cal_df["Education"].between(5, 6)]
college_test = global_test_df[global_test_df["Education"].between(5, 6)]


The following code goes through each of our demographic groups, and verifies that the training, calibration, and testing splits looks correct. Importantly, we want the splits to have the expected percentage of observations, diabetes rates to be consistent, and that we have enough observations to train a good model.

In [9]:
# Store each of the demographic datasets for verification
demographic_datasets = {
    "Female": (female_train, female_cal, female_test),
    "Male": (male_train, male_cal, male_test),
    "Younger Adult": (younger_train, younger_cal, younger_test),
    "Middle Age": (middle_train, middle_cal, middle_test),
    "Older": (older_train, older_cal, older_test),
    "Lower Income": (lower_income_train, lower_income_cal, lower_income_test),
    "Middle Income": (middle_income_train, middle_income_cal, middle_income_test),
    "Higher Income": (higher_income_train, higher_income_cal, higher_income_test),
    "Below High School": (below_hs_train, below_hs_cal, below_hs_test),
    "High School": (hs_train, hs_cal, hs_test),
    "Some College or Higher": (college_train, college_cal, college_test)
}

# Display dataset sizes and diabetes rates
for group, (train, cal, test) in demographic_datasets.items():
    print(f"\n{group}")

    for name, data in [("Training", train), ("Calibration", cal), ("Testing", test)]:
        print(
            f"{name}: {len(data)} observations | "
            f"Diabetes rate: {data['Diabetes_binary'].mean() * 100:.2f}%"
        )


Female
Training: 84986 observations | Diabetes rate: 13.01%
Calibration: 28558 observations | Diabetes rate: 12.91%
Testing: 28430 observations | Diabetes rate: 12.91%

Male
Training: 67222 observations | Diabetes rate: 15.11%
Calibration: 22178 observations | Diabetes rate: 15.25%
Testing: 22306 observations | Diabetes rate: 15.23%

Younger Adult
Training: 22837 observations | Diabetes rate: 2.88%
Calibration: 7753 observations | Diabetes rate: 3.29%
Testing: 7654 observations | Diabetes rate: 3.20%

Middle Age
Training: 56096 observations | Diabetes rate: 10.98%
Calibration: 18464 observations | Diabetes rate: 10.71%
Testing: 18562 observations | Diabetes rate: 10.81%

Older
Training: 73275 observations | Diabetes rate: 19.64%
Calibration: 24519 observations | Diabetes rate: 19.73%
Testing: 24520 observations | Diabetes rate: 19.65%

Lower Income
Training: 22611 observations | Diabetes rate: 23.90%
Calibration: 7467 observations | Diabetes rate: 23.95%
Testing: 7510 observations | D

The dataset has been split into training (60%), calibratio (%20), and testing (20%) sets. Seperate datasets were created for each of the demographic groups across sex, age, income, and education. With the results above we verified that the rates of each split to ensure that the groups would be properly represented. These datasets can now be used to begin training the models.